## Text Loader

In [ ]:
### Load docx file
from langchain_community.document_loaders import TextLoader


In [ ]:
# load text file
file_path = "../data/input/business_requirements.txt"
textloader = TextLoader(file_path)
tdocs = textloader.load()

print("Number of Pages:", len(tdocs))
print("Metadata:", tdocs[0].metadata)                   ##Print Metadata
print("Page Content:", "\n", tdocs[0].page_content)     ##Print Page Content

In [ ]:
### whitespace normalization function
import re

def clean_text(text):
    # Normalize Windows/Mac line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    
    # Remove excessive blank lines (but keep single newline)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    
    # Remove trailing spaces on lines
    text = re.sub(r"[ \t]+$", "", text, flags=re.MULTILINE)
    
    return text.strip()

for doc in tdocs:
    doc.page_content = clean_text(doc.page_content)

print("Number of Pages:", len(tdocs))
print("Metadata:", tdocs[0].metadata) 
print("Cleaned Page Content:", "\n", tdocs[0].page_content)     ##Print Cleaned Page Content


## PDF Loader

In [1]:
### Document loader for PDF files
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyMuPDFLoader

/Users/maymach09/Documents/GenAI09/MacOS/RAG/RAG_PIPELINE/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Load pdf file
file_path = "../data/input/prompting-guide-101.pdf"
loader = PyMuPDFLoader(file_path)
documents = loader.load()

print("Number of Pages:", len(documents))
print("Metadata:", documents[9].metadata)
print("Page Content Preview:\n", documents[9].page_content[:500])

Number of Pages: 68
Metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'creationdate': '2024-10-11T12:21:56-04:00', 'source': '../data/input/prompting-guide-101.pdf', 'file_path': '../data/input/prompting-guide-101.pdf', 'total_pages': 68, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-10-11T12:22:09-04:00', 'trapped': '', 'modDate': "D:20241011122209-04'00'", 'creationDate': "D:20241011122156-04'00'", 'page': 9}
Page Content Preview:
 10
You owe a response to a question, which you believe is best answered by a document in your Drive. You prompt 
Gemini in the Gmail side panel. You type:
Generate a response to this email and use @[file name] to describe how the [initiative] can complement 
the workstream outlined in [colleague’s name]’s message. (Gemini in Gmail)
Gemini in Gmail returns a suggested email that pulls directly from your own Doc. After reading it over, you select 
the Copy icon in 

## PDF Cleaning

In [3]:
### Cleaning Function for PDF
import re

def remove_page_number(text):
    return re.sub(r'^\s*\d+\s*\n', '', text)

def normalize_quotes(text):
    text = text.replace("’", "'")
    text = re.sub(r"\b(\w+)\s+s\b", r"\1's", text)
    return text

# def remove_header_footer(text):
#     # Remove header title
#     text = re.sub(r"SAMPLE BUSINESS REQUIREMENT DOCUMENT", "", text)

#     # Remove page number patterns like: 9 | P a g e
#     text = re.sub(r"\d+\s*\|\s*P\s*a\s*g\s*e", "", text)

#     return text.strip()

def fix_unicode(text):
    # Normalize unicode
    text = text.encode("utf-8", "ignore").decode("utf-8")

    # Replace weird smart quotes
    text = text.replace("͞", '"').replace("͟", '"')

    # Remove stray combining characters
    text = re.sub(r"[^\x00-\x7F]+", " ", text)

    return text


def normalize_structure(text):
    # Fix broken words across lines
    text = re.sub(r"-\n", "", text)

    # Merge lines that break mid sentence (lowercase start)
    text = re.sub(r"\n(?=[a-z])", " ", text)

    # Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Merge lines that break mid-sentence
    text = re.sub(r'\n(?=[a-z])', ' ', text)

    # Remove excessive spaces
    text = re.sub(r' +', ' ', text)

    return text.strip()

def fix_possessives(text):
    # Fix: ] s  → ]'s
    text = re.sub(r"\]\s+s\b", "]'s", text)

    # Fix standard word possessive
    text = re.sub(r"\b(\w+)\s+s\b", r"\1's", text)

    return text

def fix_contractions(text):
    text = re.sub(r"\bI m\b", "I'm", text)
    text = re.sub(r"\b(\w+)\s+t\b", r"\1't", text)
    return text

In [4]:
def clean_pdf_pipeline(text):
    text = fix_unicode(text)
    text = remove_page_number(text)
    text = normalize_quotes(text)
    text = fix_possessives(text)
    #text = remove_header_footer(text)
    text = normalize_structure(text)
    text = fix_contractions(text)
    return text.strip()

In [5]:
for doc in documents:
    doc.page_content = clean_pdf_pipeline(doc.page_content)

print("Cleaned Preview:\n")
print(documents[9].page_content[:500])
print("\nMetadata:\n", documents[9].metadata)
print(len(documents[9].page_content))

Cleaned Preview:

You owe a response to a question, which you believe is best answered by a document in your Drive. You prompt 
Gemini in the Gmail side panel. You type:
Generate a response to this email and use @[file name] to describe how the [initiative] can complement the workstream outlined in [colleague's name]'s message. (Gemini in Gmail)
Gemini in Gmail returns a suggested email that pulls directly from your own Doc. After reading it over, you select the Copy icon in the side panel and paste it directly i

Metadata:
 {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'creationdate': '2024-10-11T12:21:56-04:00', 'source': '../data/input/prompting-guide-101.pdf', 'file_path': '../data/input/prompting-guide-101.pdf', 'total_pages': 68, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-10-11T12:22:09-04:00', 'trapped': '', 'modDate': "D:20241011122209-04'00'", 'creationDate': "D:20241011122156-04'00'", 

## PDF Splitting

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=20000,
    chunk_overlap=200
)

all_chunks = []

for doc in documents:
    page_chunks = splitter.split_text(doc.page_content)

    for idx, chunk in enumerate(page_chunks):
        new_doc = Document(
            page_content=chunk,
            metadata={
                **doc.metadata,
                "chunk_id": f"page_{doc.metadata['page']}_chunk_{idx}",
                "chunk_index": idx
            }
        )
        all_chunks.append(new_doc)

print(f"Total chunks created: {len(all_chunks)}")
print("Sample chunk metadata:", all_chunks[50].metadata)
print("Sample chunk content preview:\n", all_chunks[50].page_content[:500])

Total chunks created: 68
Sample chunk metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'creationdate': '2024-10-11T12:21:56-04:00', 'source': '../data/input/prompting-guide-101.pdf', 'file_path': '../data/input/prompting-guide-101.pdf', 'total_pages': 68, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-10-11T12:22:09-04:00', 'trapped': '', 'modDate': "D:20241011122209-04'00'", 'creationDate': "D:20241011122156-04'00'", 'page': 50, 'chunk_id': 'page_50_chunk_0', 'chunk_index': 0}
Sample chunk content preview:
 This provides a helpful starting point, but you want to try getting an even better response. You click Refine and Formalize.
Gemini in Gmail: [Generates refined email suggestions] 
Gemini in Gmail
You re happy with the email, so you click Insert. You read the message one last time, make final light edits directly, and then send the message. Now, you want to learn more about the customer a

In [7]:
## Validate chunks

import random

for chunk in random.sample(all_chunks, 5):
    print("----")
    print("Page:", chunk.metadata["page"])
    print("Chunk ID:", chunk.metadata["chunk_id"])
    print(chunk.page_content[:500])

----
Page: 61
Chunk ID: page_61_chunk_0
Startup leaders
You thrive in fast-paced, dynamic environments where you can wear many hats and make a tangible impact. You re driven by a passion for innovation, a desire to learn and grow, and a tolerance for risk. Your work is unique in its variety, its potential for high reward, and its direct connection to the company's success. You re not just executing tasks; you re building something from the ground up, shaping the future of your company, and potentially disrupting entire industries.
Gem
----
Page: 27
Chunk ID: page_27_chunk_0
Frontline management
As a frontline worker manager, your team's work is indispensable to your organization your team may not primarily complete its day's work on a computer, but communication and collaboration remains key. 
This section provides you with simple ways to integrate prompts in your daily tasks. 
Getting started
First, review the general prompt-writing tips on page 2 and the Prompting 101 section at the 

## Embeddings

In [ ]:
# Initialize embedding model

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()  # Loads GEMINI_API_KEY from your .env if present

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",  # no "models/" prefix
    google_api_key=os.getenv("GEMINI_API_KEY"),  # or omit this if env var is set
)

print("Embedding model initialized successfully.")


Embedding model initialized successfully.


In [ ]:
## Create Chroma vector store and persist locally

from langchain_chroma import Chroma  # modern import

# Create Chroma vector store and persist locally
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",       # folder to store the DB
    collection_name="prompting_guide",    # any name you like
)

print("Chroma vector store created and persisted automatically.")
print(f"Documents in collection: {vectorstore._collection.count()}")

Chroma vector store created and persisted automatically.
Documents in collection: 68


In [13]:
## Load from vector store

from langchain_chroma import Chroma

# Load existing Chroma store
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings,
    collection_name="prompting_guide"
)

print(f"Loaded {vectorstore._collection.count()} documents from Chroma.")


Loaded 68 documents from Chroma.


## Retriever


In [18]:
### Test Retriever


query = "Explain Conduct fundraising and investor relations use cases from this guide."
docs = vectorstore.similarity_search(query, k=3)

for i, d in enumerate(docs, 1):
    print(f"\n=== Result {i} ===")
    print(f"Page: {d.metadata.get('page')}, Chunk: {d.metadata.get('chunk_id')}")
    print(d.page_content[:400], "...")


=== Result 1 ===
Page: 60, Chunk: page_60_chunk_0
You gathered useful information from your discussion with Gemini Advanced. You want to go deeper in your brainstorming around two competitors in particular. You type: 
Generate a competitive analysis of [company] versus [competitor] within the current market landscape. 
(Gemini Advanced)
You select Share & export and Export to Docs.
Use case: Conduct fundraising and investor relations 
You re read ...

=== Result 2 ===
Page: 12, Chunk: page_12_chunk_0
Example use cases
Analyst and public relations
Use case: Prepare for analyst or press briefings
You need to create a brief to prepare a spokesperson for an upcoming meeting with analysts and the media for a new product launch. You open a new Doc and prompt Gemini in the Docs side panel. You type:
Generate a brief template to prepare [spokesperson] for an upcoming media and analyst briefing for 
@[ ...

=== Result 3 ===
Page: 10, Chunk: page_10_chunk_0
Communications
As a communications pr